# 02 — Second Spark: same LAN, SSH to **both**

**Milestone:** two Sparks each have a **management LAN** address and you can SSH to **both** from your workstation.

You still are **not** relying on the QSFP cluster link yet—that is [`03`](03_qsfp_interconnect_link_and_mtu.ipynb).

**Why this matters:** you will copy images, keys, and repos using this path before the high-speed fabric is debugged.

## Checklist

- [ ] Second Spark: power + **Cat6 to same router/switch** (or VLAN you can route).
- [ ] Discover **SPARK2_LAN_IP** (DHCP table or static).
- [ ] Verify SSH to the other Spark from your current admin host.
- [ ] (Recommended) Same **username** and **SSH key** on both for less friction.

**Green signal:** you can reach both Spark management hosts from your workflow (laptop or head), and this notebook auto-skips self-target checks.

In [ ]:
import shutil
import socket
import subprocess

USER = "cesarb-ai"  # Replace with your SSH username
SPARK1_LAN_IP = "192.168.0.28"
SPARK2_LAN_IP = "192.168.0.29"  # second Spark


def local_ips() -> set[str]:
    ips: set[str] = {"127.0.0.1"}

    # Hostname-based resolution
    try:
        _, _, host_ips = socket.gethostbyname_ex(socket.gethostname())
        ips.update(host_ips)
    except Exception:
        pass

    # Linux-friendly source of all assigned addresses
    try:
        r = subprocess.run(["hostname", "-I"], capture_output=True, text=True)
        if r.returncode == 0:
            ips.update(r.stdout.split())
    except Exception:
        pass

    return ips


def probe(ip: str, my_ips: set[str]) -> None:
    ssh = shutil.which("ssh")
    if not ssh:
        print("❌ ssh not found in PATH")
        return

    if ip in my_ips:
        print(f"⏭️  {ip} is local to this notebook host; skipping self-SSH probe.")
        return

    r = subprocess.run(
        [ssh, "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", f"{USER}@{ip}", "hostname"],
        capture_output=True,
        text=True,
    )
    if r.returncode == 0:
        print(f"✅ {ip} -> {r.stdout.strip()}")
    else:
        print(f"❌ {ip} -> {(r.stdout or r.stderr).strip()}")
        print("   Hint: this check is key-based (BatchMode=yes). Manual password SSH may still work.")


my_ips = local_ips()
print("Notebook host IPs:", ", ".join(sorted(my_ips)))
probe(SPARK1_LAN_IP, my_ips)
probe(SPARK2_LAN_IP, my_ips)

## Next

Install the **QSFP** cable between Sparks (follow NVIDIA playbook for **which ports**). Then open **`03_qsfp_interconnect_link_and_mtu.ipynb`**.